In [48]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

In [49]:
df = pd.read_csv('../data/processed_data.csv')
df.head()

,order_status,customer_state,price,freight_value,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,...,approval_hours,carrier_handover_hours,purchase_month,purchase_dayofweek,distance,product_volume,same_state,freight_ratio,product_density,package_size
0,delivered,SP,29.99,8.72,4.0,500.0,19.0,8.0,13.0,housewares,...,0.178333,56.795833,10,0,0.176608,1976.0,1,0.281381,0.252908,40.0
1,delivered,SP,29.99,8.72,4.0,500.0,19.0,8.0,13.0,housewares,...,0.178333,56.795833,10,0,0.176608,1976.0,1,0.281381,0.252908,40.0
2,delivered,SP,29.99,8.72,4.0,500.0,19.0,8.0,13.0,housewares,...,0.178333,56.795833,10,0,0.176608,1976.0,1,0.281381,0.252908,40.0
3,delivered,BA,118.70,22.76,1.0,400.0,19.0,13.0,19.0,perfumery,...,30.713889,11.109167,7,1,7.660024,4693.0,0,0.190142,0.085215,51.0
4,delivered,GO,159.90,19.22,1.0,420.0,24.0,19.0,21.0,auto,...,0.276111,4.910278,8,2,4.627149,9576.0,0,0.119453,0.043855,64.0


In [50]:
df.dtypes

order_status                         str
customer_state                       str
price                            float64
freight_value                    float64
product_photos_qty               float64
product_weight_g                 float64
product_length_cm                float64
product_height_cm                float64
product_width_cm                 float64
product_category_name_english        str
seller_state                         str
payment_type                         str
payment_installments             float64
payment_value                    float64
delivery_time_days                 int64
approval_hours                   float64
carrier_handover_hours           float64
purchase_month                     int64
purchase_dayofweek                 int64
distance                         float64
product_volume                   float64
same_state                         int64
freight_ratio                    float64
product_density                  float64
package_size    

In [51]:
# Splitting data into X and y
X = df.drop("delivery_time_days", axis=1)
y = df["delivery_time_days"]

In [52]:
# Feature Encoding
from sklearn.preprocessing import LabelEncoder
categorical_columns = X.select_dtypes(include="object").columns
le = LabelEncoder()

for column in categorical_columns:
    X[column] = le.fit_transform(X[column].astype(str))

In [53]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [54]:
# Feature Scaling
from sklearn.preprocessing import LabelEncoder, StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline Model Training

In [55]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

models = {
    "Linear Regression": LinearRegression(),
    "KNN": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "SVR": SVR(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

In [56]:
results = []

In [57]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
for name, model in models.items():

    if name in ["Linear Regression", "KNN", "SVR"]:
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_test_scaled)

    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = root_mean_squared_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    results.append([name, mae, rmse, r2])

baseline_results = pd.DataFrame(results,columns=["Model", "MAE", "RMSE", "R2"])

baseline_results.sort_values("R2", ascending=False)

,Model,MAE,RMSE,R2
4,Random Forest,3.646659,5.379120,0.623483
5,Gradient Boosting,4.259769,6.012558,0.529586
3,SVR,4.486908,6.538656,0.443662
1,KNN,4.848348,6.769318,0.403718
0,Linear Regression,4.942910,6.829770,0.393020
2,Decision Tree,4.985607,7.722200,0.224032


# Hyperparameter Tuning

#### Linear Regression

In [58]:
tuned_lr = LinearRegression()
tuned_lr.fit(X_train_scaled, y_train)
lr_pred = tuned_lr.predict(X_test_scaled)

print("Linear Regression")
print("R2:", r2_score(y_test, lr_pred))

Linear Regression
R2: 0.39302038182456145


#### K Nearest Neighbor

In [59]:
knn_params = {
    "n_neighbors":[3,5,7,9,11],
    "weights":["uniform","distance"],
    "p":[1,2]
}

In [60]:
from sklearn.model_selection import GridSearchCV

tuned_knn = GridSearchCV(
    KNeighborsRegressor(),
    knn_params,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

In [61]:
tuned_knn.fit(X_train_scaled, y_train)
print(tuned_knn.best_params_)

{'n_neighbors': 11, 'p': 1, 'weights': 'distance'}


In [62]:
knn_best = tuned_knn.best_estimator_
knn_pred = knn_best.predict(X_test_scaled)
print("KNN R2:", r2_score(y_test, knn_pred))

KNN R2: 0.5010424925404586


#### Decision Tree

In [63]:
dt_params = {
    "max_depth":[5,10,15,20,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}

In [64]:
tuned_dt = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    dt_params,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

In [65]:
tuned_dt.fit(X_train, y_train)
print(tuned_dt.best_params_)

{'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10}


In [66]:
dt_best = tuned_dt.best_estimator_
dt_pred = dt_best.predict(X_test)
print("Decision Tree R2:", r2_score(y_test, dt_pred))

Decision Tree R2: 0.5198726677577707


#### Random Forest

In [70]:
rf_params = {
    "n_estimators":[100,200],
    "max_depth":[10,20,None],
    "min_samples_split":[2,5],
    "min_samples_leaf":[1,2]
}

In [71]:
from sklearn.model_selection import RandomizedSearchCV

tuned_rf = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=rf_params,
    n_iter=20,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

In [72]:
tuned_rf.fit(X_train, y_train)
print(tuned_rf.best_params_)

{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': None}


In [73]:
rf_best = tuned_rf.best_estimator_
rf_pred = rf_best.predict(X_test)
print("Random Forest R2:", r2_score(y_test, rf_pred))

Random Forest R2: 0.6272490975727827


#### Gradient Boosting

In [74]:
gb_params = {
    "n_estimators":[100,200],
    "learning_rate":[0.01,0.05,0.1],
    "max_depth":[3,5,7]
}

In [75]:
tuned_gb = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_params,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

In [76]:
tuned_gb.fit(X_train, y_train)
print(tuned_gb.best_params_)

{'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200}


In [77]:
gb_best = tuned_gb.best_estimator_
gb_pred = gb_best.predict(X_test)
print("Gradient Boosting R2:", r2_score(y_test, gb_pred))

Gradient Boosting R2: 0.6309920941159022


# Model comparison after hyperparameter tuning

In [81]:
final_results = pd.DataFrame({

    "Model":[
        "Linear Regression",
        "KNN",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting",
    ],

    "R2":[
        r2_score(y_test, lr_pred),
        r2_score(y_test, knn_pred),
        r2_score(y_test, dt_pred),
        r2_score(y_test, rf_pred),
        r2_score(y_test, gb_pred),
    ]
})

final_results.sort_values(by="R2",ascending=False)

,Model,R2
4,Gradient Boosting,0.630992
3,Random Forest,0.627249
2,Decision Tree,0.519873
1,KNN,0.501042
0,Linear Regression,0.393020


In [82]:
import joblib
joblib.dump(gb_best, "../model/delivery_time_model.pkl")

['../model/delivery_time_model.pkl']